In [9]:
#!/usr/bin/env python3
import cv2
import pytesseract
import numpy as np
from PIL import Image
import os

image_path = input()

def diagnose_ocr_issues(image_path):
    """Systematic diagnosis of OCR problems"""
    
    print(f"=== DIAGNOSING OCR ISSUES FOR: {image_path} ===\n")
    
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print("❌ ERROR: Could not load image")
        return
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    
    print(f"📏 Image dimensions: {width} x {height} pixels")
    
    # 1. Check image resolution
    print("\n1. RESOLUTION CHECK:")
    if height < 1000:
        print("❌ PROBLEM: Low resolution! OCR needs 300+ DPI")
        print(f"   Current: ~{height}px height (needs 3000+ for good OCR)")
        print("   SOLUTION: Use higher resolution scan or upscale image")
    else:
        print("✅ Resolution looks adequate")
    
    # 2. Check image quality metrics
    print("\n2. IMAGE QUALITY CHECK:")
    
    # Check contrast
    contrast = gray.std()
    print(f"   Contrast score: {contrast:.1f}")
    if contrast < 30:
        print("❌ PROBLEM: Low contrast - text may be too faint")
    else:
        print("✅ Contrast looks good")
    
    # Check for blur
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    print(f"   Sharpness score: {laplacian_var:.1f}")
    if laplacian_var < 100:
        print("❌ PROBLEM: Image appears blurry")
    else:
        print("✅ Image sharpness looks good")
    
    # 3. Check document skew
    print("\n3. DOCUMENT SKEW CHECK:")
    coords = np.column_stack(np.where(gray > 0))
    if len(coords) > 0:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        
        print(f"   Detected skew angle: {angle:.2f}°")
        if abs(angle) > 2:
            print(f"❌ PROBLEM: Document is skewed by {angle:.1f}°")
            print("   SOLUTION: Apply rotation correction")
        else:
            print("✅ Document alignment looks good")
    
    # 4. Test different Tesseract configurations
    print("\n4. TESSERACT CONFIGURATION TEST:")
    
    test_configs = {
        "Default": "",
        "Table Mode": "--psm 6",
        "Uniform Block": "--psm 6 --oem 3",
        "Single Column": "--psm 4",
        "Sparse Text": "--psm 11",
        "Numbers Only": "--psm 8 -c tessedit_char_whitelist=0123456789.,$ "
    }
    
    results = {}
    for name, config in test_configs.items():
        try:
            text = pytesseract.image_to_string(gray, config=config).strip()
            confidence = pytesseract.image_to_data(gray, config=config, output_type='dict')
            avg_conf = np.mean([int(c) for c in confidence['conf'] if int(c) > 0])
            
            results[name] = {
                'text': text[:100] + "..." if len(text) > 100 else text,
                'confidence': avg_conf,
                'length': len(text)
            }
            
            print(f"   {name:15} | Confidence: {avg_conf:5.1f}% | Length: {len(text):4d} chars")
            
        except Exception as e:
            print(f"   {name:15} | ERROR: {str(e)[:50]}")
    
    # 5. Identify best configuration
    print("\n5. RECOMMENDATIONS:")
    if results:
        best_config = max(results.items(), key=lambda x: x[1]['confidence'] * (1 + x[1]['length']/1000))
        print(f"✅ BEST CONFIG: {best_config[0]} (Confidence: {best_config[1]['confidence']:.1f}%)")
        print(f"   Sample text: {best_config[1]['text']}")
    
    # 6. Preprocessing recommendations
    print(f"\n6. PREPROCESSING RECOMMENDATIONS:")
    print("   Try this preprocessing pipeline:")
    
    # Show what preprocessing would help
    processed = preprocess_for_ocr(gray)
    cv2.imwrite('debug_preprocessed.jpg', processed)
    print("   - Saved preprocessed version as 'debug_preprocessed.jpg'")
    
    # Test on preprocessed version
    try:
        preprocessed_text = pytesseract.image_to_string(processed, config="--psm 6")
        preprocessed_conf = np.mean([int(c) for c in pytesseract.image_to_data(processed, config="--psm 6", output_type='dict')['conf'] if int(c) > 0])
        
        print(f"   - Preprocessed confidence: {preprocessed_conf:.1f}%")
        print(f"   - Preprocessed text length: {len(preprocessed_text)} chars")
        
    except Exception as e:
        print(f"   - Preprocessing test failed: {e}")

def preprocess_for_ocr(gray):
    """Apply standard preprocessing for printed documents"""
    
    # 1. Resize if too small
    height, width = gray.shape
    if height < 2000:
        scale = 2000 / height
        gray = cv2.resize(gray, (int(width * scale), 2000), cv2.INTER_CUBIC)
    
    # 2. Denoise
    denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    
    # 3. Improve contrast with CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(denoised)
    
    # 4. Correct skew
    coords = np.column_stack(np.where(enhanced > 0))
    if len(coords) > 0:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        
        if abs(angle) > 0.5:  # Only correct if significant skew
            center = (enhanced.shape[1]//2, enhanced.shape[0]//2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            enhanced = cv2.warpAffine(enhanced, M, (enhanced.shape[1], enhanced.shape[0]), 
                                    flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    # 5. Apply adaptive thresholding for clean text
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                  cv2.THRESH_BINARY, 11, 2)
    
    return binary

if __name__ == "__main__":
    # Replace with your image path
    image_path = input() # Change this to your actual image file
    
    if os.path.exists(image_path):
        diagnose_ocr_issues(image_path)
    else:
        print(f"❌ Image file not found: {image_path}")
        print("Please update the image_path variable with your actual file path")

=== DIAGNOSING OCR ISSUES FOR: C:\Users\gcyan\Nova\nova_data\trial.pdf ===

❌ ERROR: Could not load image
